In [ ]:
%autosave 300
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%config Completer.use_jedi = False

Autosaving every 300 seconds


In [2]:
import os

os.chdir("..")
print(os.getcwd())

/mnt/batch/tasks/shared/LS_root/mounts/clusters/soutrik-vm-dev/code/Users/Soutrik.Chowdhury/lightning-template-hydra


In [3]:
!ls

Dockerfile   data.dvc		      image.jpg        scripts
README.md    docker-compose-adv.yaml  logs	       src
__pycache__  docker-compose.yaml      notebooks        test_metrics.md
checkpoints  dvc.lock		      notes.md	       tests
configs      dvc.yaml		      pytest.ini       train_acc.png
data	     hello.py		      pytorch_project  train_loss.png


In [4]:
import os
import glob
import yaml
import pandas as pd
import json
from datetime import datetime

# Paths
base_path = "./logs/train/multiruns"
output_path = "./artifacts"

In [7]:
def multirun_artifact_producer(base_path: str, output_path: str):
    latest_folder = max(glob.glob(os.path.join(base_path, "*")), key=os.path.getmtime)
    # Initialize JSON structure
    output_data = {}

    # Process each run directory
    for run_dir in os.listdir(latest_folder):
        run_path = os.path.join(latest_folder, run_dir)
        if os.path.isdir(run_path):
            # Paths to files
            hparams_path = os.path.join(run_path, "csv", "version_0", "hparams.yaml")
            metrics_path = os.path.join(run_path, "csv", "version_0", "metrics.csv")

            # Read hparams.yaml
            with open(hparams_path, "r") as file:
                hparams = yaml.safe_load(file)

            # Read metrics.csv and calculate averages
            metrics_df = pd.read_csv(metrics_path)
            avg_train_acc = metrics_df["train_acc"].dropna().mean()
            avg_val_acc = metrics_df["val_acc"].dropna().mean()

            # Create JSON structure for this run
            output_data[f"run{run_dir}"] = {
                "hparams": hparams,
                "metrics": {"avg_train_acc": avg_train_acc, "avg_val_acc": avg_val_acc},
            }

    # Save to JSON
    os.makedirs(output_path, exist_ok=True)
    output_file = os.path.join(
        output_path, f"aggregated_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    )
    with open(output_file, "w") as json_file:
        json.dump(output_data, json_file, indent=4)

In [8]:
multirun_artifact_producer(base_path, output_path)